# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. See: https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print('Dataset Title:', metadata.name)
print('Description:', metadata.description)
print('\nCollection Timeframe:', getattr(metadata, 'dataCollectionTimeframe', None))
print('License:', metadata.license)
print('Keywords:', metadata.keywords)

# Show available record sets
print('\nRecord Sets (@id):', getattr(metadata, 'recordSet', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

This dataset contains multiple tables as described in the metadata. We will attempt to enumerate the record sets. If the metadata object does not directly contain the record set list, we will retrieve it using the `dataset.record_sets()` method.

In [ ]:
# Get all record sets
record_sets = list(dataset.record_sets())
print('Discovered Record Sets:')
for rs in record_sets:
    print(f"@id: {rs.id}, Name: {rs.name}, Description: {rs.description}")

# For each record set, list fields (@id, name, description, dataType)
for rs in record_sets:
    print(f"\nFields for Record Set @id: {rs.id} ({rs.name})")
    for f in rs.fields:
        print(f"  Field @id: {f.id} | Name: {getattr(f, 'name', '')} | dataType: {getattr(f, 'dataType', '')} | Description: {getattr(f, 'description', '')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All entities are referenced by their `@id` field. Adjust the code below dynamically according to discovered record set and field `@id`s.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record Set {record_set_id}, Columns: {df.columns.tolist()}")
        print(df.head(), '\n')
    else:
        print(f"Record Set {record_set_id} has no records.")

# Choose a record_set for EDA: eg. the first one if exists
selected_record_set_id = record_set_ids[0] if record_set_ids else None

if selected_record_set_id:
    print(f"Selected for EDA: {selected_record_set_id}")
    print(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations include removing outliers, transforming distributions, or grouping data by key attributes.

We will select a numeric field and a categorical/grouping field based on the field list above.

In [ ]:
# Identify numeric and group fields from the selected record set
numeric_field_id = None
group_field_id = None
rs_obj = [rs for rs in record_sets if rs.id == selected_record_set_id][0] if selected_record_set_id else None
if rs_obj:
    numeric_candidates = [f.id for f in rs_obj.fields if getattr(f, 'dataType', '').lower() in ['integer', 'float', 'number']]
    group_candidates = [f.id for f in rs_obj.fields if getattr(f, 'dataType', '').lower() in ['text', 'string', 'boolean']]
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None
    group_field_id = group_candidates[0] if group_candidates else None

print(f"Numeric Field @id: {numeric_field_id}")
print(f"Group Field @id: {group_field_id}")

df = dataframes[selected_record_set_id]
# Example filter: filter records above a threshold for the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by the group_field_id
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset using matplotlib.

In [ ]:
import matplotlib.pyplot as plt

# Simple visualization: histogram of numeric field, grouped boxplot by categorical field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    df[numeric_field_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing the clinical and genetic FAIR^2 dataset using `mlcroissant`.
By referencing entities via their `@id` and using automated handling, we efficiently loaded and visualized patient-level data. 
Key takeaways include: 
- Small cohort size (N=3), emphasizing rare disease context.
- Each field and table is referenced by its `@id` ensuring clarity and reproducibility.
- Dataset is structured for clinical genotype-phenotype analysis and not for predictive AI modeling without augmentation.

For deeper analysis, consult detailed field documentation via metadata or schema URL and tailor further EDA to the clinical questions of interest.